# 数据清洗与预处理

本教程将学习数据清洗和预处理的基本操作。

## 1. 导入必要的库

In [11]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

## 2. 加载含有缺失值的数据

In [12]:
# 创建含有缺失值的示例数据
data = {
    'date': ['2024-01-01', '2024-01-02', '2024-01-03', '2024-01-04', '2024-01-05'],
    'product': ['A', 'B', 'A', 'C', 'B'],
    'quantity': [10, np.nan, 15, 20, 12],
    'price': [100, 200, np.nan, 150, 180],
    'region': ['北京', '上海', np.nan, '深圳', '广州']
}

df = pd.DataFrame(data)
print('原始数据:')
print(df)

原始数据:
         date product  quantity  price region
0  2024-01-01       A      10.0  100.0     北京
1  2024-01-02       B       NaN  200.0     上海
2  2024-01-03       A      15.0    NaN    NaN
3  2024-01-04       C      20.0  150.0     深圳
4  2024-01-05       B      12.0  180.0     广州


## 3. 检测缺失值

In [13]:
print('缺失值检测:')
print(df.isnull())

print('每列缺失值数量:')
print(df.isnull().sum())

缺失值检测:
    date  product  quantity  price  region
0  False    False     False  False   False
1  False    False      True  False   False
2  False    False     False   True    True
3  False    False     False  False   False
4  False    False     False  False   False
每列缺失值数量:
date        0
product     0
quantity    1
price       1
region      1
dtype: int64


## 4. 处理缺失值

In [14]:
# 方法1：删除含有缺失值的行
df_dropna = df.dropna()
print('删除缺失值后的结果:')
print(df_dropna)

# 方法2：填充缺失值
df_fillna = df.copy()
df_fillna['quantity'] = df_fillna['quantity'].fillna(df_fillna['quantity'].mean())
df_fillna['price'] = df_fillna['price'].fillna(df_fillna['price'].median())
df_fillna['region'] = df_fillna['region'].fillna('未知')

print('填充缺失值后的结果:')
print(df_fillna)

删除缺失值后的结果:
         date product  quantity  price region
0  2024-01-01       A      10.0  100.0     北京
3  2024-01-04       C      20.0  150.0     深圳
4  2024-01-05       B      12.0  180.0     广州
填充缺失值后的结果:
         date product  quantity  price region
0  2024-01-01       A     10.00  100.0     北京
1  2024-01-02       B     14.25  200.0     上海
2  2024-01-03       A     15.00  165.0     未知
3  2024-01-04       C     20.00  150.0     深圳
4  2024-01-05       B     12.00  180.0     广州


## 5. 处理重复值

In [15]:
# 添加重复行
df_duplicate = pd.concat([df_fillna, df_fillna.iloc[[0]]])
print('含有重复值的数据:')
print(df_duplicate)

# 检测重复值
print('重复值检测:')
print(df_duplicate.duplicated())

# 删除重复值
df_unique = df_duplicate.drop_duplicates()
print('删除重复值后的结果:')
print(df_unique)

含有重复值的数据:
         date product  quantity  price region
0  2024-01-01       A     10.00  100.0     北京
1  2024-01-02       B     14.25  200.0     上海
2  2024-01-03       A     15.00  165.0     未知
3  2024-01-04       C     20.00  150.0     深圳
4  2024-01-05       B     12.00  180.0     广州
0  2024-01-01       A     10.00  100.0     北京
重复值检测:
0    False
1    False
2    False
3    False
4    False
0     True
dtype: bool
删除重复值后的结果:
         date product  quantity  price region
0  2024-01-01       A     10.00  100.0     北京
1  2024-01-02       B     14.25  200.0     上海
2  2024-01-03       A     15.00  165.0     未知
3  2024-01-04       C     20.00  150.0     深圳
4  2024-01-05       B     12.00  180.0     广州


## 6. 数据类型转换

In [16]:
print('原始数据类型:')
print(df.dtypes)

# 转换日期类型
df_fillna['date'] = pd.to_datetime(df_fillna['date'])
# 转换为类别类型
df_fillna['product'] = df_fillna['product'].astype('category')
df_fillna['region'] = df_fillna['region'].astype('category')

print('转换后的数据类型:')
print(df_fillna.dtypes)

原始数据类型:
date         object
product      object
quantity    float64
price       float64
region       object
dtype: object
转换后的数据类型:
date        datetime64[ns]
product           category
quantity           float64
price              float64
region            category
dtype: object


## 7. 异常值检测与处理

In [17]:
# 添加异常值
df_outlier = df_fillna.copy()
df_outlier.loc[0, 'quantity'] = 1000  # 添加异常值

print('含有异常值的数据:')
print(df_outlier)

# 使用IQR方法检测异常值
Q1 = df_outlier['quantity'].quantile(0.25)
Q3 = df_outlier['quantity'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print(f'异常值检测边界:')
print(f'下限: {lower_bound}, 上限: {upper_bound}')

# 标记异常值
df_outlier['is_outlier'] = (df_outlier['quantity'] < lower_bound) | (df_outlier['quantity'] > upper_bound)
print('标记异常值:')
print(df_outlier)

# 处理异常值（替换为中位数）
median_quantity = df_outlier['quantity'].median()
df_outlier.loc[df_outlier['is_outlier'], 'quantity'] = median_quantity
print('处理异常值后的结果:')
print(df_outlier.drop('is_outlier', axis=1))

含有异常值的数据:
        date product  quantity  price region
0 2024-01-01       A   1000.00  100.0     北京
1 2024-01-02       B     14.25  200.0     上海
2 2024-01-03       A     15.00  165.0     未知
3 2024-01-04       C     20.00  150.0     深圳
4 2024-01-05       B     12.00  180.0     广州
异常值检测边界:
下限: 5.625, 上限: 28.625
标记异常值:
        date product  quantity  price region  is_outlier
0 2024-01-01       A   1000.00  100.0     北京        True
1 2024-01-02       B     14.25  200.0     上海       False
2 2024-01-03       A     15.00  165.0     未知       False
3 2024-01-04       C     20.00  150.0     深圳       False
4 2024-01-05       B     12.00  180.0     广州       False
处理异常值后的结果:
        date product  quantity  price region
0 2024-01-01       A     15.00  100.0     北京
1 2024-01-02       B     14.25  200.0     上海
2 2024-01-03       A     15.00  165.0     未知
3 2024-01-04       C     20.00  150.0     深圳
4 2024-01-05       B     12.00  180.0     广州


## 8. 数据标准化

In [18]:
# 数据标准化
df_standardized = df_fillna.copy()
df_standardized['quantity_std'] = (df_standardized['quantity'] - df_standardized['quantity'].mean()) / df_standardized['quantity'].std()
df_standardized['price_std'] = (df_standardized['price'] - df_standardized['price'].mean()) / df_standardized['price'].std()

print('标准化后的数据:')
print(df_standardized[[ 'quantity', 'quantity_std', 'price', 'price_std']])

标准化后的数据:
   quantity  quantity_std  price  price_std
0     10.00     -1.128330  100.0  -1.560213
1     14.25      0.000000  200.0   1.084216
2     15.00      0.199117  165.0   0.158666
3     20.00      1.526564  150.0  -0.237999
4     12.00     -0.597351  180.0   0.555330
